# 2. Feature Engineering (Engenharia de Características)

Neste notebook, focaremos exclusivamente na Engenharia de Características (Feature Engineering) para selecionar e transformar as variáveis do conjunto de dados de preços de casas nos EUA, preparando os dados para a modelagem preditiva.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Configurações de visualização
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

## 2.1 Carregamento e Limpeza Prévia dos Dados

Para que este notebook seja executável de forma independente, vamos realizar o carregamento do dataset `train.csv` e aplicar o tratamento de valores nulos estruturado na fase de Limpeza de Dados.

In [ ]:
# Carregando o dataset de treino
df_train = pd.read_csv('train.csv')

# Tratamento de nulos conforme data_description.txt
cols_na_is_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
                   'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                   'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                   'MasVnrType']

for col in cols_na_is_none:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna('None')

cols_na_is_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 
                   'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF','TotalBsmtSF', 
                   'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_na_is_zero:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna(0)

# Preenchimento da LotFrontage com a mediana do bairro
df_train['LotFrontage'] = df_train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Preenchimento com a moda para demais categóricas residuais
df_train['Electrical'] = df_train['Electrical'].fillna(df_train['Electrical'].mode()[0])

print(f"Dados originais carregados e limpos com formato: {df_train.shape}")

## 2.2 Remoção de Outliers

Conforme identificado na análise bivariada, há dois outliers na variável `GrLivArea` (Área útil acima do solo) com mais de 4000 pés quadrados que foram vendidos por preços muito abaixo da média. Eles serão removidos do conjunto de treino.

In [ ]:
# Removendo outliers extremos de GrLivArea
outliers = df_train[(df_train['GrLivArea'] > 4000) & (df_train['SalePrice'] < 300000)].index
df_train = df_train.drop(outliers)
print(f"Dataset após remoção de {len(outliers)} outliers: {df_train.shape}")

## 2.3 Criação de Novas Features (Feature Creation)

Criamos novos atributos de valor estratégico para modelos de preço imobiliário:
- **TotalSF**: Área útil total somada do porão, primeiro e segundo andar.
- **TotalBathrooms**: Quantidade consolidada de banheiros e lavabos.
- **HouseAge**: Idade do imóvel no momento da venda (ano da venda menos o ano de construção).
- **RemodAge**: Anos passados desde a última reforma no momento da venda.
- **OverallQual_GrLivArea**: Relação de interação entre a qualidade dos acabamentos e o tamanho habitável da casa.

In [ ]:
# Construção de novas features
df_train['TotalSF'] = df_train['TotalBsmtSF'] + df_train['1stFlrSF'] + df_train['2ndFlrSF']
df_train['TotalBathrooms'] = df_train['FullBath'] + 0.5 * df_train['HalfBath'] + df_train['BsmtFullBath'] + 0.5 * df_train['BsmtHalfBath']
df_train['HouseAge'] = df_train['YrSold'] - df_train['YearBuilt']
df_train['RemodAge'] = df_train['YrSold'] - df_train['YearRemodAdd']
df_train['OverallQual_GrLivArea'] = df_train['OverallQual'] * df_train['GrLivArea']

df_train[['TotalSF', 'TotalBathrooms', 'HouseAge', 'RemodAge', 'OverallQual_GrLivArea']].head()

## 2.4 Tratamento de Assimetria e Transformações

A variável target `SalePrice` é altamente assimétrica para a direita. Para linearizar as correlações com as preditoras e melhorar o ajuste de modelos de regressão, aplicaremos a transformação logarítmica $\log(x+1)$ no preço e nas principais colunas de tamanho (`LotArea` e `GrLivArea`).

In [ ]:
# Aplicando transformação logarítmica np.log1p
df_train['SalePrice_Log'] = np.log1p(df_train['SalePrice'])
df_train['LotArea_Log'] = np.log1p(df_train['LotArea'])
df_train['GrLivArea_Log'] = np.log1p(df_train['GrLivArea'])

# Visualizando a distribuição de SalePrice antes e depois da transformação
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df_train['SalePrice'], kde=True, ax=ax[0], color='blue')
ax[0].set_title('Distribuição Original de SalePrice')
sns.histplot(df_train['SalePrice_Log'], kde=True, ax=ax[1], color='green')
ax[1].set_title('Distribuição Transformada (Log1p) de SalePrice')
plt.show()

## 2.5 Codificação de Variáveis Categóricas (Encoding)

Para preparar os atributos de texto/categóricos para algoritmos matemáticos:
1. **Variáveis Ordinais**: Variáveis de qualidade e condição que possuem uma ordem natural explícita são mapeadas para inteiros entre 0 e 5.
2. **Variáveis Nominais**: Aplicaremos One-Hot Encoding para obter representações binárias das categorias restantes.

In [ ]:
# Ordinal Encoding para colunas de qualidade/condição
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for col in qual_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].map(qual_map)

# Separando as variáveis preditoras (X) e o target transformado (y)
y = df_train['SalePrice_Log']
X = df_train.drop(columns=['Id', 'SalePrice', 'SalePrice_Log'])

# One-Hot Encoding nas demais variáveis qualitativas nominais
X_encoded = pd.get_dummies(X, drop_first=True)
print(f"Formato dos preditores após codificação dummy: {X_encoded.shape}")

## 2.6 Padronização das Features

Aplicaremos o `StandardScaler` do `scikit-learn` para deixar as features numéricas em escala comum (média zero e variância unitária). Isso previne que variáveis em grande escala dominem o aprendizado de modelos e distâncias no espaço vetorial.

In [ ]:
# Padronização usando StandardScaler
scaler = StandardScaler()
numeric_cols = X_encoded.select_dtypes(include=[np.number]).columns.tolist()

X_scaled = X_encoded.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_encoded[numeric_cols])

print(f"Features padronizadas com sucesso! Novo formato do dataset: {X_scaled.shape}")